# 📋 MNT02 미션 02 — 결측치 보정, 계절성 수요 분리 및 수리부속 품목코드(NSN) 정제 전처리 모듈 구현

**Student 학습용** / CMS 편성 2시간 / Level 3

---

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다.

폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'MNT02_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 4개: MNT02_Colab_실행안내.md, MNT02_M02_forms.md, MNT02_M02_lesson.md, MNT02_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVz+D1bE0QYAANcMAAAbAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVdNTxtXFN3zK55UdVkH8lW6RCmqLAFFcbLpBpngSG7B'
    'tmyzSFcGBuRgU0ywwziMydCYzzrN4A88qGbjnzPvzX/oue8920PCAhkzb+4799xzz718x2bnXow/ZM+Sy9FFJgp1//1bJip5vt4eG5sIqacLmezqUiyRDcVT'
    'bxKL/KTHxEHba9lM1HP8xmBiMycOSsKw9HGEEdsdv2zwY0vYNX/bDf0ZT4kaTpoNfnrLC3leqIeYsA3RbAvbEkd7vOWy38LzXqvHvE6DXxt+5WJwznMdcdxm'
    'Im8Ku8L83TbfKfPtMhNHDYXYr7T5rsH4zr/ifVvsO+IshyS2AUKHGHsYYn7FEB/KfsVivNYTtTZ3i/wj4DT4WZEJoy5uzMCdidWV1Jt+NxVNLEUz/W7mVZy+'
    '4uOPePaH5Vg0neh3V6LZ1HIyuxxflLmtN8SHi0AIUXX5dY5IRYoA5+BS33D63an5MPPX6/2uODqR/B04XpOeDjGa32bwKMT4F1dsmmwmlskkcf3rZHolI2om'
    'EeabBvNahqjcSii1WyoPOBU1l04QNttgfrHEC+cAEEa9GD879lwXh3vyHbsSxK5AT4UZ8Ijm536XFzq8lQPm5j9MtPPQSL/rdXvelcN4MSeOkRdvtUXVwJGO'
    'Jf7eBKe2v1Gj2PTguD0KP/Z4kAxvGVoGbCn2mqQVX0ktx1agtmg2nkww3nDFUQl6I8TQiTCuUG2WjqWWo69iC69XE6/onLB6EBtC5cBpMA/U+ZQg1SgMELsO'
    'WPGcd6IGgi0bPN8jtNsy37dIYaHFGFiOUWiVAy+ZUvhHJr/8j5cswsebFVxgWPxyk6KCEZSdl4ZR55LZ8CCp2NJ0Op1MM7Gx5m9YqmuklKWOSQKivocajV4f'
    'e0LS7YhCAx/n6IFBuf1Dg6VXEwsr8UyGGKAaNk1RK7JhUzDICh1FtH5dAmKGIgxSvi57Tg6ncl7zFkrqofqSUEkF0gNPe/jEU8gsGPfG8DqlQFzUVQMkGMVz'
    'aRfGFd8+kXfVc0S+rtY9naq7XLapUsh1Dk0T7PUhM09DLLmaTa1mMw9meTPvm+4D/0NROOeklUx2CQ/RstmlWDpNyAmySqLr8tNevwv2Qr9niDoQAKUC5ahq'
    'fHcTVsX45Tnfz8u6OjlUDUBK/FORsvKcNXzRb6Jn+ZeczKpypKHLzpME6qzkRXh/t6oZkNHtwXFSuL0GE4HKQIzUxR1uvFbH3+roKsmCHzTgYQOQw/cJzEV7'
    'UE3eLAsLPcCbBrVaTRadH9cCTP4Ip/+RibM1Ye8B4Sa3TwKuPsjcrqDa6C8UHDd/9is2cnjkXbVJTHTCa+b8LTsoMviOZUM+Zl6swyFQIr9qgi6mrINo9Kvg'
    'xQVkaqyZ6annc9PPF8Jz8y9f4OISImi49xkJo8AFi0XCsy9npl5M/0zKhFSk6FQ2yvOG/mCJa+s+fx04nnJxWVdYxmkNtDmW+ETh9mSH4r2DrYAvT4K6SVT6'
    '0fj491Qxf/ctENEfuOMQlXeGnbTZagXTSlQhKdVNhKrqjh5JcekhKFmBPnihQXer3yiVr2vIqFEqEJx7LJVGzdyhjPx3eX55IecEBgRVAanQr7DGC2eY9+EN'
    'KlEm2XjOlSjYw0DAXDF4oTpI+CckPDFO8Uh2O672eSo++JIXq9ZRYwB8QD9kLnfCYK60hbkmtQUg+9bEuOcAxtGJ7Bo9M1su/BWAMSDJdbqW17yQcwaskHNS'
    'mfVMGEni2oJ80UJUDcrcOVfLAp3DUDbQvt/CoTlDUu80MMpkLutSReg3sg29YKy3IQ5qMtyHaIplgx4p6Y/qPTYxLueJnMG5iXE9uwcqyJskpoA6Dku0AWkq'
    'Zx/3u7NP8TOpJjPJQ/d0ZHpm+hmEvjAbjkTCv85FSEcUXEbUDScFpPUhBQSnF1s72rugbBNdJPEbw/Yb8vdsNiL3kLz5QMPRpaIdaaAsmbN6NZgzNkZ+lifv'
    'OnsrDQSH9fBSQ4EGJcWUo1NTopZOhZfEKdtW8UnmAr+WNgDRw7gpXb9YxNLCKAfV6ncsZBU7WXQJI4F20cHiiR2P738OzL5BidXs1bsOFhn0QEBgENMFHiNZ'
    'eFelSlutlDObn4pEAKxT8tpQ0vaN3LMM7Rn3+4uepMPYtIbJujfkq8F1We+4cknTjge63tVJgNyuc+cwYEBjSjtgbiPPfolnmf+Xq7X5cDL0BGbkOUZAaqiv'
    'bLGtIi0T6rFenOQ8uZaDh+9foRxyyfhGIQPLQAQshyr+SBXKN4KCkv9b8I+Ov4FLDNoeUWwQLbtBDi21tanVSgXX/5CgxzFytrErS1lqgxo5TdmRtn5m3N0M'
    '/gdQSwMEFAAAAAgAAAAhXM30qd6DBwAAMBIAABIAAABNTlQwMl9NMDJfZm9ybXMubWTdWEtPG1cU3vtXXCmq1EoQSNIVmyrLSA2bdldVCCWkihQgBbqI5MUA'
    'Y+RgR5jgIQMZW+PEhEeMMtjGHiTTBT8l2c298x/6nXPvtU0etIu2UqskGN/neXznO9/NNXF38sfxm+Iu/qnqpnJPhNreVIWKeO+URdKMVFxTZ76QrbYKvRGR'
    'tFwV5nlZ3le7ZSE7rnzTEDLaoBH8KjuOWnsm0ud5eXSozstyK/h68ofJbwT2qzDAh6uaPu85OpBbeZGcNlLfzWTUy5JsxSLdeJpEzoT4KfVcnPjzpxvYNHNa'
    '0g3U2Y64Ma5e5YSqtNXqMqaW1e7hCC1JvUOyNWk66k1FpGunSaunggOsH/8KzsTtJH4KF/KqE7CLtfawixhJIhenClk4UPnADOCnLNSF8usqOoCtHLiKo/Yd'
    'ke54qhKn23m5XsbM9UxGtpykeS7o3jCYELfvqB1HyHex3DqRtYpAZC669LWKGK+Eah8uNQ/JWQpHIx4xHsqWK0wUkm4vOYlGbATSl66QuR5vqTa0p6riWlM8'
    'P2mFbH9AY75aiYS1qRnBLLnXE/LMTU5LqRfAL9h87Zo5HKaxx/yJ4GQyWRvxrDFwzJobuISTbN/Kkzayh+8aOZRFWHrRNanImuDjZADo6AATkUPrsrhidHRU'
    'fPEn5sWlv/+hAYrs7TtwPUp3n5HLgYyCfkwwDEhkMqNCrTQQL5EWS0hcvy4LgQpLE+L9do0OkYVT5JEhGUcUXp4wo+pFW1aLjA29G4cCei9KQu7XkjjGgpxs'
    'tAmLVDRBOEG3hkHSOk29EHtK8nXxoqvLF59hoLxzgIWSDkgB3QY9tA9VC6RRmSg/jwIEjswsn48y3l/m2t1r00Y4l3rHE4QlWXTSHXAIiCJcJuwU6kwRu2XV'
    'PDYAE9Iv8bbQUdW9IZzptdbhK9daC1Fjh2VYchXKMlfkECTFpgG1fC2qvm8zG0CFp0sFzCMLeaonoaNoq86YaHM+qMHIVz7Hjg7YcU0U5IZrgDE4kUIHBCi3'
    'Lo9yfZ/1LcoN01Wqr34OC/V0G3WW5opqvZ6ux4ZALoVFI40TzHCg/QHYyMF+8HzoMQPIPQ78aUBkq+3FzqKq9LDMxPqfKN/MLbCJYXeOcNFRtWg4wuyrkM18'
    '6scUTBX9roO8SphPujFsI36W+yWiw9QDg5+BcNtoFDhXrvg2j8T+K5EN21B+qKusnw5lQOeTwY78rO8hxMWirPTA5qmrw8sJp9K0hSSbLn3tz3GBfjynqVJc'
    'smAI9zr8Js82XX9D2Jn2dQ+GQ6ru8LXmNhBAJyCvdXtGBI9RtUzjvDR6PsTkcYSeMShR7QHG+7grNEBJuu8VGhS3ElE/+jAdnbUF8O+0AiI+ONnO9xNhvlKF'
    'tfPKy5uKNilgprQpy8UoKNlZ1o2yFSqX6LwbyzfwV9YC/GpiN7QbAFKVIokAlfNV0MM6IV2faFk3QeoIllzRl5O2Q3SRroGYwa6rge4BqAJifv51mDF4RK2f'
    'obQ/OGVZIKijQOBXpYceYvPwwfE45al3qgoNo8L6PQiJ9g50A1fVnAyBbjiBT6MruNG/FZZcNEmIIdgOuDcKkmaDygSCCQzSX2zhcCUVP5hdmppbnMMI/4Ge'
    'ehcbU+mi4gFTn3uC+htAkVuR58rCTp8gyFb3BO3N2MfY7pBGu1SKPMcUYxTRgHIpVDuB3NBSbN9lvz/q3FmzRGRtmLRNQ7XATg/o0mRUC6yrMY+ZG/idlDpI'
    'CgJiAm4dIxjmeuLxvyzYR74o0Jnr8j7cM43+Y/1N4dEpMIKRKJbE+m6Zs2zgolmAOiVJT3gMTbN7SAinXGkxQ1kwGYb12GkwRkZ753SR7nFWxdp+pfWRbloX'
    'XQgbRLPaF5c6OmymbpxWjn9nQJQVN20g2Y8Jsbhwb2xm6dHY7NzS+M2p+9NL01P3Hs1Mzy0+nPvl+uMnFFpjJ8Kj1YW2SVuKOkMErQhZFO/XnnMp4zGSbgYE'
    'AJomfVAtQZrZlj05PzcDbObfYvjGLWpieD5xqyEhJlLfw0PMEASSA+UNtWRioe8il1l+Wwg4pm+b1KDIEQfVOVS1E5vO5ltdCbZ2NU2BhTeDMU1d2GJEzCBy'
    'qClUyPczi4vznCuuHV06dAOjW4MJcDBa5tO437Jx50T9Tx55prb7gDdtEBIFskXT3ZmPKOs2Z/PHTWZMc2GfdwztDIe9iCU4xK+nq6TGoICqJfue00VOWccu'
    'FdSowNTrHgqJ8EH8Q3FFYlCTlgk/j5uBbPXlYXQ5aDyrY2M0GLsDIX+Ms7GDgqK89YE8YnXie8K+HlV9k3oMKa2NHVIQ3ZChipcAvV47rnIrRonhcv0c5kLY'
    'iszZjFy8LuKaSMs9QIWeFqhe7B3SgdSyyf29MS2PSDfCek26hHPuopZsYVe1oUP/Gah+a6CKhdCWf0oRI2Jh5vH8wtKiWdCfm/r1t+lHD5eeXJ+9PyIezC/M'
    'Lo7xf7lM4fAp/o4ZqihdQvxa0xrkVU7LKn4k6fTotoFpdk+LPyKaFxGYX5Z864RVVqQx0A9Yv4N93IAG7BX6QhYKWkswt3mAd1rNE28Vgk/i8gdQSwMEFAAA'
    'AAgAAAAhXFjzHYctCQAAWhgAABMAAABNTlQwMl9NMDJfbGVzc29uLm1k7VjfTxvZFX73X3GlqFIrORDISl2hVlHUrqpITVS1fdqqAkrYNmoC20Ar7dsAQ+Rg'
    'UgzxwBjGdOiaBBKvOvgHtlXzwp+yeZt753/od865Y49D0qRvq2ofQmB8fe853/nOd74719Tde7+9Manu4t+3TlnFjch0j0zPV7rZMqGXV3HTNWHBuGfKFHyz'
    'X1b63NUv6kpHW/QEv+pzxzx5ppKdgn51ai7K+nnww3u/ufcjhe+bMMB/rmn4/J1XJ/p5QcXteuK7udy1a8qsrSSV07jZx2enyXaQy8WRg4c4xcGGSj+LTLWV'
    'uBEda9yq0sWWLvbMYUkfBcYNbFCXHV08Mf9c5/3WgsuOOazHzVDFnX58Funjvv164p3qYkEXa2PKNL5BHCruRvqoaqr+x6eef2+qdBCWJrtPE8/XG+UrqZuq'
    'a7NPvIDCEBAOSrrZtd9UulE2gWsqjkq2ngKNXO56uiLxXCxS+l9ds+6Pye4tZYo1s9E2eyXgwYfaCNJUB98HLGsF9YsHyyr5e5fSmjTFII7cN06Z9ugexdEO'
    '0FbGc3XRf+N4w5Nlzyllatu62U7BQ8qAN68AgUp8zxSrBETcCZJKmRJAoPr5GfDlFXHjlNHTq6F5GUjJAlclT9qm2s8rU93EKZcdkAFRM7f+8dTWQmGblE7Y'
    '3vQqauIG17vaQskJVhOumP1T4M5177biLr58XjDngak66bqeD9oo89Ix+xF+ItMNICZ8AJirkTKHx8YLaZXuIUzse1CmDYZcQETAzAR93urMxVZ0JGVTwAM3'
    'BBJ2EfHa+DUTnTAFe27cLtnCZ5ojOXCJhqYQAgo6h0rPDIvbkW666dmIGpgQxcIAhUsZdF39Tnjx+6udxj39ftwsaHlaAr5QdnHDMS+qiiPpm+AE62/8ID8K'
    'KBflqJW/CgrIY3GQB/jJtWQMwGXAuw00Cfqk4plqN9ktoAD4ZCzDtdUIh+h6f0otPZ4bn19+OP5oYfnG5PT92eXZ6bmH87MLSw8W/jj25VciQpIeEDMvC3xu'
    'M0QnSGeBoERXyMXXm9RI+AupcRjU1l5Br7YuOz9bfDj7B+oj6r/0obSV0uvdZKOrz1fSJ0nZJe0Jq3isPr/zK46cpWBKzXwg4Jkx9HDbFOv478RWU/QAu4cO'
    'yGcJbBqv03iIHrfvECnMOSGFFJ24cTHS3wOZnFIj4c1QBOO//uz2z+9+Nvbo/gzT8KSPo8CACyRjt7j5Y1AIGQD7Okh02Zn4NH1wUALXLIKXncmJOAqYXc9Q'
    'qMN1HR5bTYM2jkpOMQDKyuxuQxaADM+ZacyZ6S8WHz9aomi4fPcX55bGH89/ufh4eYlkmMSt4nJ1pEhJEeSQalZbulGw8kdrLZbpPLmuks1NaMlH1IHWcgWw'
    '9otHy9MLSwszTA/XeBdTLFfExnpXLalvn+woO5UwoUzN4Y/xGUAxe2C8XyLZvbe4MI+2LrzG44mbBBhGo3QrcVD0MZWNVCXTcvNZ0D6WyMFQQpUCiJAzbHUd'
    '+aS/w7Fo1qoMktCJR45QnhlJFbLz7u4EITb3p/m5Pw+X2eP/NvvwATCaH9JxpJQjMDMfmZ7uGR/ELCUSiCoMFDjCH6OUVSQpo1OylQpgKo7D+YxmQ1vfvqP0'
    'pgOkMC+aXSiGnekkVMmTEG3j1/SxzxMac4J0LQiZ5qsncW8lTYLok3gVauikEuitCj25Zn+fEOvTdJP9ZyzZ3AQgUwlilstZWRKDghqhOKRSDDA5DeoMzql6'
    '8Y7Z446OGxInGzHXnBfstvTuv+PG5gDCMO5t3qI9JYYMLFCCvZLSL4/ibtdahTQGkpAA9L3siLaCLKe64clw9DIVzaQurs/KC2dka5XLvUeorgoUjBPNbMLc'
    'ho9Ndmp2V8AFWIfGh8JPZ/UgqxF+TNlvjrgHlB9/HvrWPVgzYUHM2+FGw9IOQGFJfjAdiGLrff4KrCEPOYrJTiHJKTUkFZ+cQBrTMLvs/M6CeFP4I7wEW81R'
    'xHge1gUhUMiykxEWUciKfbowg8jbo9WezsOywoZE7OoBGkPMAhCCPX11Qip5GkneSFt4+JOfimUjHWNJ0SVfBu8tajHj1vSrdRBIKEAmZq16Mz5rZQ1WSjIO'
    'lS07p5qCh9pXEGaxDeD0Fv1Wo8mRIvwu9n0i/qQdkCURqkv/WQMw6D1xwiwqNYeUgEsj0pFGLfP5bcmwbiC1TedBlnVgVLHOzCIoQ2+whdnoUQPJbcJyf2TA'
    'DdwqmjlZc8hZrwCglELiWrlM7ZIJjsB2Zb7uQ455oNLWsGphVp/fMwIysoqqXhWX1HaO+NvVb6j2nm81ZzjnR+0poq5tU48TZbYqBHIn5KmzWueG4TsTYhdK'
    'KelAhvJ5xj7D3hWAH5xHH83HPf+2HRD5bWE3SpAdJsFRRVeUUOTvsfwfsRQqc69UXHSl7QyyQB9jfWas37JrBh9P/+WvcALLX5E5wzI2ataGoQvzmZmM2zAE'
    'BUlXt9Hxee7hVgF/Z4Uun04IABl3qB6drn7RpzmVmflZ5x96dOeTZPjmw+LLw4C1jacvo01CRnd0ooZ7hoFi9cPKMG2KveOWQ67Hxi2WIdPVOAJKRWUWimT1'
    'NwqYWgwvl4klwfo38oHFd7LCOlWWNDYsvtU6SsfO5bRYE2P83kWMx9R35I0EzWe6T+6XaY6LxR/CLj2JCsIioQ9RBb1xzF4/LBEdrZemzj5cJxEPj62p5hHC'
    'Y2YwMMjO0Uq2W1K+yw5mf+CaQzcVCkGHw5QapTfGW7lJix+H/8HLIiFqwwMq2QEsAb5xyt9N2//GIQ9nzk/N0VlaxcZrsd/SYKiL7TBEOi5dhq9YtRsARtqE'
    'vvvl/NLSIpeIzbkMUzqBu8ReHPZKuGoYfyUD900LN5dl6v/jrYOVhgG9rcrA1sD8iJzxHAA7OVopG4vduGjd4P5j7WEW7e/H2kePNR4qlP7x+NB9Inr7Bhf0'
    'tvc98Zn8htea5iFDPxGGimh/UBDy6sNjMC9DcPzq2wvqH2kYno8IiDU/jnYQI9tPqQqpWY9YwFnJ/ZRkZS+CvGdc+DvMKGkNrCsepEfIgTyJcAYRgJTMA6uT'
    'w4IdSkM4cpnXvv/lre+4bCgvfwfvfsfsFV7uZMhW7oN6tYVqy0PSH7n9bpRTxRrYlP8AUEsDBBQAAAgIAAAAIVyxBwko6AYAAJ8NAAAcAAAATU5UMDJf6rO1'
    '7Ya1X+yekeyEseyViOuCtC5tZG1WXU8bRxR9968YKe+YkI+mjyihLVJAUT4UqS8RTWmCRHEERFX7tIY1MrZTTPDGa7LrLI2NgS7qYhtsFPPin+OZ/Q89d2bW'
    'Xj4kPizvzp1zzz333HuLzcw+H59gg9ZpuHHKRHVbmCdMWFm+1k4kBoEh1tNMZG2+7/MzQ2x8YPxDINx2aAZ8z6EnYrck7Kzo7vU7ouoPWh7jZ6Yw3X6H5w/E'
    'PxmGCOG6wwZNI9zw8FLuXLhdNugigCtck4l8TeROQ8vmuVIsugwS2hYLSyauEp4b5rqimuH5LM/Xxpj4XOStrnBtFn7M8qPDfmf22Wy/88Ozh7gDsM5tiQAp'
    'UNzBaVE4e2L3kKmc4oFwv+cAtcELRr8zaJnCukCMckY025RUnZ8DZbXIvxZ4vcfoqFcHXtEwwFQO4HWoROLWLTY5zcKKg5skNMsEFrDbE56VSNweY4vzKyup'
    'JVzTDUQN5y9KfMeRYd0LSR7lhJ82olj4FFpgec0HSEBxbe53+x1cIRxTVE3GmyV8QrSmIfZdxoND3rSIVH6AGxHeS4/gMZ4/5S0j9pi3a8Lp8UNctu2IvBNa'
    'h1EqE2MsgpivhZ82Ub8AaRBS3mqLvTYIILyT06Jc1Oz0O6ENWL7OCoA9R3H5uRjVt2UMmhey7N7lG6M6kJ6axzjT/DcGGGhRiBi+O2NSfJ4VozAsFITbIzw8'
    'sIWdDq02kPkqgdhNxN/XHuLhFnmYCdML1112Z3CCE26bBOKaQB3lRaJYC5BfpiByNRJihdQi9oIYpLuAVLX50TdeVHKChKEBoqti6rwk+cMSq3bTTOp0YkUC'
    'IXlnqIMRUY4H6SmVexZU3mrzug1E4tShhlPJS4mc27JgaweD83QE817UOyDV4l82dZ0RTxVAFRph/+vyvC//o7vlTdQ78ka86RYYPcnYTJQDnCHEqqljUF++'
    '/TP5U+qP5Mu3c6sov8qcXgt3P4wuDBweOMmhQagswkKRDAQEQtmDk2DUseRLaV12SXDkGbAaF3wKF8reyvD9C61dKSDHpFYaNNfxLj8MNHgqc9YRpoNqksto'
    'qbThh6AZ/VtB/ygHo+Qhh62K8EwQw4sk6MGpDxHgg7DwZWXYaVLmI2aVynjTxFnqVtlKuZIOXy2SS0lONapyUTTSwttW1MZrrysn9Tpzt9+ZuY/fB3yLIsOv'
    'yCrARbpNIZAVgGXtS+qXJqUQKIdqw2cTCRUYUNez7MeFVRb+3dVUTjwYuwcdDgLzmvsycn569JF4RALdPfoMS9lzOBhc82+ySdihJxs374AVpI4+yIpPbSTB'
    '0b4oT7fAv/gYOVIiZD5hxSbbKBd5KcA7uIQeSUx4HcXAuUaWGgKRYoZ3dAhjY0rJyGXQ6ao6XxMqoqPHMSZ8xJTeVOkCI3qPUIFFyS4bfVXvUf/nvZvnAMiF'
    'ZJPU+p6VVNVKhlublCzuxh0kgDOTHx1QpKPMFSuEWMsUlyYv2kYNUrq57pE9kdayEDEGEwYN+rFgkNQkISzcPVDq0tNbNW2sdlppeV9brh7icDTdNLttMs8o'
    'zLD/b5yZGL4stLsoEORCmipnuA+iGtuy8Qw13fWg4sGneBtDiGobwDnJ+ZkTCXHmO9n5st9IrbBxQHk2PfPi8eTzqUdaeHFEul+OQ8vTIxplQj2vtBnF8mhE'
    'RmPceTw1+XR26umr6dknL55H2jXp8FWRUB2yGBbqEoxihBTlNglrtF7gxjNTeUw00dCTWAeqGJOw/X03quDILaCCHaA+w4RwKJrKVxXVhRd5k9OkQcUGzQPI'
    '8KbGgo1isA0tlLA0alIeEC9VCjKmSrmkCBJPtQ7kugg0vXUd9bIm+/f4ci6yrkPLN0R5I657vcvFNrafp5+A7IepxblfyJJsn3u18KOBnHimi+f8LK284/KI'
    'xsTE4A0/mxxt3TB4Y1NkDOVp798tpuZ+XUnKxXXsr4V3OM8bBb5zPDwMxJerrkYlih1tJlQ805HUhJVS5DPDHQcVkSSRJLnfi+lsYWlldfn969XUMhNBl5xM'
    'jxgDr9IBSAJrGtL/5g/HDxE4hINaji6lE5J9SiJmKNf3SorSRAVlaaPNV5o0dY+aNIlEnPqhhcccXMobmz72pDjbUKRHywq1BqiS7+rjtMJlEUStduqeawrW'
    'mWwUyEjkaeIxD2s+J0J0hobqq3jXeg4mLeaNxO3jr9qtNLVq586VVlLvl1/Pv0otL7xZWJpbXEm+WVh9tTz/2/zy/BK+nxifuH97fPx70scl/yey1rH4HDyd'
    'mnw0M4XAKouF39+llldp6O6coMV1Ulenm1retXtTgeXeRYts0b46bV3j9jjtMfXe1XnLoi+caOOw7+BNNQzQXphRVHtYwHbx6qqX+B9QSwECFAMUAAAICAAA'
    'ACFc/g9WxNEGAADXDAAAGwAAAAAAAAAAAAAApAEAAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kUEsBAhQDFAAAAAgAAAAhXM30qd6DBwAAMBIAABIA'
    'AAAAAAAAAAAAAKQBCgcAAE1OVDAyX00wMl9mb3Jtcy5tZFBLAQIUAxQAAAAIAAAAIVxY8x2HLQkAAFoYAAATAAAAAAAAAAAAAACkAb0OAABNTlQwMl9NMDJf'
    'bGVzc29uLm1kUEsBAhQDFAAACAgAAAAhXLEHCSjoBgAAnw0AABwAAAAAAAAAAAAAAKQBGxgAAE1OVDAyX+qzte2GtV/snpHshLHslYjrgrQubWRQSwUGAAAA'
    'AAQABAAUAQAAPR8AAAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 상황: 어떤 문제를 해결하나요?

**미션 개요**

수리부속 청구/출고 원시 기록에 정제 규칙 10종 이상을 적용하고, 월별 수요에서 계절성 성분을 분리하여 전처리 데이터의 정합성 검증 통과율을 확보하는 실습입니다.

**배경 및 필요성**

현장 실무 데이터에는 중복 청구, NSN 형식 오류, 음수 수량, 결측치 등 다양한 오류가 포함되어 있으므로, 정확한 수요 예측을 앞서 철저한 정제와 표준화 과정이 필수적입니다.

전산 장애로 발생한 결측 월은 단순 보간보다 오차가 적은 계절성 보간을 적용하여 데이터의 신뢰성을 높여야 합니다.

**핵심 과업**

- 전처리 규칙 10종 이상 적용: 중복 청구 제거, NSN 형식 정규화 및 미등록 NSN 격리, 날짜·단위 통일, 음수 및 과다 수량 보정 등을 수행하며 긴급 불출은 이상치로 삭제하지 않고 보존합니다.
- 결측치 보정 및 비교: 전산 장애 월에 대해 계절성 보간을 수행하고 단순 선형 보간과의 보간 오차(MAE)를 비교·분석합니다.
- 기술통계 및 계절지수 산출: 정제 전후의 기술통계를 비교하고, 주요 품목군(예: 축전지 등)의 월별 계절지수를 도출하여 수요 변동 특성을 해석합니다.

**검증 및 산출물**

**검증 기준:** 정제 규칙 10종 이상 적용 완료, 정합성 검증 통과율 100% 달성, 긴급 불출 보존 여부 및 계절성 보간 오차가 단순 보간보다 작은지 확인.

**주요 산출물:**

src/etl/mnt02_data_cleansing.py 및 정제 품질 보고서 (규칙별 건수, 전후 통계, 계절지수).

## 📁 이번 미션에서 완성할 것

- **미션에서 완성할 내용**
  - 중복 청구 제거, NSN 형식 정규화와 미등록 NSN 격리, 날짜·단위 통일, 음수·과다 수량 보정 등 정제 규칙 10종 이상을 적용하고 긴급 불출은 이상치로 지우지 않는다.
  - 전산 장애로 빠진 월은 계절성 보간으로 채우고 단순 선형 보간과 오차를 비교한다.
  - 정제 전후 기술통계와 품목군별 계절지수를 제시한다.
- **구현 계약**
  - 핵심 구현 계약 src/etl/mnt02_data_cleansing.py / fmt_nsn: NSN 원문 s → 가상 표준 NSN 문자열 또는 None. 숫자13자리를 제공 형식으로 정규화하고 원문·격리 기록은 유지한다.
- **변경 과제와 검증**
  - 추가 변경·검증 청구 기록의 결측치 보정, 계절성 수요 분리, 품목코드(NSN) 정제를 수행하는 전처리 모듈을 구현한다.
  - AI와 미등록 NSN·미래 날짜 격리 원문, 규칙별 처리 근거, 정제 후 논리 재검증을 확인하고 단위 환산 변경 결과를 비교한다.
  - 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `src/etl/mnt02_data_cleansing.py`, `reports/mnt02_cleansing_quality.md`, `forms/MNT02_M02_forms.md`


---

## 🎯 미션 02. 결측치 보정, 계절성 수요 분리 및 수리부속 품목코드(NSN) 정제 전처리 모듈 구현

**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 결측치 보정, 계절성 수요 분리 및 부품코드(NSN) 정제 전처리 모듈을 구현한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **청구 기록의 결측치 보정, 계절성 수요 분리, 품목코드(NSN) 정제를 수행하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **중복 청구 제거, NSN 형식 정규화와 미등록 NSN 격리, 날짜·단위 통일, 음수·과다 수량 보정 등 정제 규칙 10종 이상을 적용하고 긴급 불출은 이상치로 지우지 않는다 과정을 진행하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/mnt02_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%

- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축

- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


## 📋 순서대로 할 활동

### 활동 1: 계획과 사전 판단

- 원문 목표 분석 및 사전 계획 수립 수리부속 청구 기록의 결측치 보정, 계절성 수요 분리, 그리고 품목코드(NSN) 정제를 수행하는 전처리 모듈의 전체적인 구현 방향을 기획합니다.
- 입력 데이터의 기간, 단위, 사용 가능 시점을 명확히 정의하고, 긴급 불출 기록을 단순 이상치로 판단하여 임의로 삭제하지 않고 원문을 안전하게 보존하는 방안을 수립합니다.
- AI 위임 범위 및 검증 계획 설정 데이터 전처리 규칙 10종 이상 적용 과정에서 AI에 위임할 코드 작성 범위와 활용할 자료를 구체적으로 확정합니다.
- 정합성 검증 통과율 100% 달성 여부와 계절성 보간법이 단순 보간법에 비해 오차(MAE)가 적은지 객관적으로 비교·검증할 구체적인 절차를 사전에 준비합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M02 계획: 청구 기록의 결측치 보정, 계절성 수요 분리, 품목코드(NSN) 정제를 수행하는 전처리 모듈을 구현한다.에 필요한 입력·기간·단위·사용 가능 시점과 fmt_nsn의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2: AI와 코드 변경

- **함수 구현과 첫 실행**
  - 핵심 함수(fmt_nsn) 구현 및 첫 실행 보존 수리부속 품목코드(NSN) 정제를 위한 핵심 함수 fmt_nsn을 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다.
  - NSN 원문 $s$를 입력받아 가상 표준 NSN 문자열 또는 None을 반환하며, 숫자 13자리를 제공 형식으로 정규화하는 동시에 원문과 격리 기록은 안전하게 유지합니다.
- **AI 활용과 변경 과제**
  - AI와 기획 반영 및 변경 과제 수행 AI와의 협업을 통해 앞서 수립한 전처리 기획안을 실제 코드에 구체적으로 적용합니다.
  - 변경 과제: AI와 함께 미등록 NSN 및 미래 날짜 격리 원문 처리, 규칙별 처리 근거, 정제 후 논리 재검증 과정을 확인하고, 상자 단위 보정 등 단위 환산 변경 결과를 상호 비교합니다.
  - 실제 코드와 실행 출력 결과물은 모두 안전하게 저장합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M02 구현: src/etl/mnt02_data_cleansing.py의 fmt_nsn를저장하고계약〔NSN 원문 s → 가상 표준 NSN 문자열 또는 None. 숫자13자리를 제공 형식으로 정규화하고 원문·격리 기록은 유지한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3: 근거 대조와 재실행

- 수정 코드 저장 및 재실행 검증 도메인 전처리 모듈(src/etl/mnt02_data_cleansing.py)에 반영된 수정 코드를 안전하게 저장하고 스크립트를 재실행합니다.
- 계절성 보간 비교 시 동일한 결측월·품목·분모 조건을 엄격히 유지하며, 정제 후 출고 <= 청구 논리 관계가 올바르게 성립하는지 검증합니다.
- AI 설명 및 코드 선택 대조 분석 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료, 실행 결과 로그와 상호 대조하여 타당성을 엄격히 검토합니다.
- 보완이 필요한 답변과 수정 사항을 다시 한번 확인하고, 긴급 불출 원문 보존 및 선형 보간·중심 이동평균의 올바른 활용 여부를 최종 점검합니다.

#### 확인할 사항:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M02 검증: [필수] 전처리 모듈 — 정제 규칙 10종 이상 적용, 정합성 검증 통과율 100%, 긴급 불출 보존, 계절성 보간이 단순 보간보다 오차가 작은지 확인했는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - 음수·오타·상자 단위 보정은 교육용 주입 원인 전제입니다.
  - 원문·격리 기록을 보존하며 긴급 불출을 단순 이상치로 삭제하지 않습니다.
  - 선형 보간·중심 이동평균은 사후 분석용이며 미래를 쓰지 않는 예측 피처와 구분합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4: 최종 판단과 산출물

- **산출물 제출과 상태 기록**
  - 종합 산출물 제출 및 상태 구분 원본 평가 기준, 변경 전후 비교 코드, 실제 AI 대화 내역, 그리고 최종 보고서를 누락 없이 체계적으로 제출합니다.
  - 미달 항목, 미측정 항목, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다.
- **미션별 검증 원칙**
  - 전처리 원칙 및 데이터 정합성 준수 음수 값, 오타, 상자 단위 보정 등의 전처리는 교육용 주입 원인에 따른 것임을 전제로 철저히 관리합니다.
  - 원문과 격리 기록을 안전하게 보존하며, 긴급 불출 기록을 단순 이상치로 판단하여 임의로 삭제하지 않습니다.
  - 선형 보간과 중심 이동평균은 사후 분석용으로만 엄격히 활용하며, 미래 시점의 정보를 참조하지 않는 예측 피처와 명확히 구분하여 적용합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M02 완료: src/etl/mnt02_data_cleansing.py, reports/mnt02_cleansing_quality.md, forms/MNT02_M02_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 02의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 직접 구현하기

다음 코드의 `TODO`를 완성해 실행합니다. 출력을 확인하지 않았다면 성공했다고 적지 마세요.


In [ ]:
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """수리부속 청구/출고 원시 기록에 정제 규칙 10종(NSN 형식 정규화, 중복 제거, 날짜 통일, 단위 환산, 결측 월 계절성 보간 등)을 적용하고, 월별 수요에서 계절성 성분을 분리한 뒤 정합성 검증 통과율을 확인한다."""
    import re
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 야전 정비창 수리부속 청구/출고·장비 가동률 데이터 생성기
    # 실제 부대·품목·재고·단가 정보와 무관하다. 품목 40개 x 48개월(가상 2022-01 ~ 2025-12) 월별 자료를 만든다.
    # ---------------------------------------------------------------------------
    FSC = {"2520": "power_train", "2530": "brake_steer", "2910": "fuel", "2930": "cooling",
           "2940": "filter", "5330": "gasket", "6140": "battery", "5961": "semiconductor"}
    EXERCISE = {3, 8, 10}      # 가상 대규모 훈련 월(훈련 계획으로 미리 알 수 있음)
    N_MONTHS = 48


    def season(fsc, month):
        """품목군별 계절 계수: 축전지는 겨울, 냉각계통은 여름에 수요가 늘어난다."""
        # [TODO : 학생 실습 2] 결측치 보정, 계절성 수요 분리 및 수리부속 품목코드(NSN) 정제 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    def make_items(n=40, seed=2):
        """품목 마스터: NSN(가상), 품목군, 장비 유형, 기준 수요, 단가, 조달 리드타임, 공급업체 수, 생산(단종) 상태."""
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n):
            f = str(rng.choice(list(FSC)))
            status = str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
            rows.append({"item_id": f"P{i + 1:03d}", "nsn": f"{f}-00-{rng.integers(100, 1000)}-{rng.integers(1000, 10000)}",
                         "fsc": f, "group": FSC[f], "equip_type": str(rng.choice(["A", "B", "C"])),
                         "base": float(np.round(rng.lognormal(2.7, 0.8), 1)),
                         "price": int(rng.lognormal(12, 1.0)) // 100 * 100,
                         "lead_time": int(rng.integers(1, 7) if status == "생산중" else rng.integers(4, 13)),
                         "suppliers": int(rng.integers(1, 5)) if status == "생산중" else 1,
                         "status": status, "alt_available": bool(rng.random() < (0.6 if status == "생산중" else 0.3))})
        return pd.DataFrame(rows)


    def make_history(items, months=N_MONTHS, seed=3):
        """월별 청구 수요(개), 긴급 불출 건수, 장비 유형별 운용시간·가동률."""
        rng = np.random.default_rng(seed)
        t = np.arange(months)
        moy = t % 12 + 1
        ex = np.isin(moy, list(EXERCISE))
        hours = {e: (100 + 20 * rng.random(months)) * np.where(ex, 1.5, 1.0) for e in "ABC"}
        avail = {e: np.clip(0.93 - 0.12 * (hours[e] - 100) / 80 + rng.normal(0, 0.015, months), 0.6, 0.99) for e in "ABC"}
        rows = []
        for it in items.itertuples():
            trend = 1 + rng.uniform(-0.1, 0.3) * t / months
            s = np.array([season(it.fsc, m) for m in moy])
            lam = it.base * s * trend * hours[it.equip_type] / 110
            d = rng.poisson(lam)
            rows.append(pd.DataFrame({"item_id": it.item_id, "t": t, "ym": [f"{2022 + k // 12}-{k % 12 + 1:02d}" for k in t],
                                      "month": moy, "exercise": ex.astype(int), "op_hours": hours[it.equip_type].round(1),
                                      "availability": avail[it.equip_type].round(3), "demand": d,
                                      "emergency": rng.binomial(d, 0.05)}))
        return pd.concat(rows, ignore_index=True)


    def make_stock(items, hist, seed=4):
        """현재(가상 2026-01) 재고: 보유량과 미입고 발주량(조달 중인 물량). 단종 품목은 추가 발주가 어려워 미입고량이 적다."""
        rng = np.random.default_rng(seed)
        recent = hist[hist["t"] >= hist["t"].max() - 5].groupby("item_id")["demand"].mean()
        st = items[["item_id", "lead_time", "status"]].copy()
        st["on_hand"] = [int(round(recent[i] * rng.uniform(0.5, 3.0))) for i in st["item_id"]]
        cover = np.where(st["status"] == "생산중", rng.uniform(0.6, 1.3, len(st)), rng.uniform(0.0, 0.6, len(st)))
        st["on_order"] = [int(round(recent[i] * L * c)) for i, L, c in zip(st["item_id"], st["lead_time"], cover)]
        return st[["item_id", "on_hand", "on_order"]]
    # ---------------------------------------------------------------------------

    items = make_items()
    hist = make_history(items)
    master = set(items["nsn"])
    nsn_of = dict(zip(items["item_id"], items["nsn"]))

    # 1) [교육용가상자료] 최근 24개월 월 수요를 개별 청구 기록으로 풀고 일부러 오류를 섞는다
    rng = np.random.default_rng(5)
    OUTAGE = {"2024-07", "2025-02"}            # 전산 장애로 기록이 올라오지 않은 월(가상)
    rows, rid = [], 300000
    for r in hist[hist["t"] >= 24].itertuples():
        if r.ym in OUTAGE or r.demand == 0:
            continue
        k = int(np.ceil(r.demand / 5))
        for j, q in enumerate(rng.multinomial(r.demand, np.ones(k) / k)):
            if q == 0:
                continue
            rid += 1
            rows.append({"req_id": f"R{rid}", "date": f"{r.ym}-{rng.integers(1, 29):02d}", "nsn": nsn_of[r.item_id],
                         "qty_req": int(q), "qty_issued": int(q), "uoi": "EA", "emergency": bool(j < r.emergency)})
    raw = pd.DataFrame(rows)
    n = len(raw)
    idx = lambda frac: rng.choice(n, int(n * frac), replace=False)
    raw.loc[idx(0.05), "nsn"] = raw["nsn"].str.replace("-", "")                       # 하이픈 누락
    raw.loc[idx(0.03), "nsn"] = " " + raw["nsn"] + " "                                  # 앞뒤 공백
    raw.loc[idx(0.04), "date"] = raw["date"].str.replace("-", ".")                      # 날짜 형식 혼재
    raw.loc[idx(0.02), "qty_req"] = -raw["qty_req"]                                     # 음수(입력 오류)
    raw.loc[idx(0.02), "qty_issued"] = raw["qty_req"].abs() + 2                         # 불출 > 청구
    bx = idx(0.02); raw.loc[bx, "uoi"] = "BX"; raw.loc[bx, "qty_req"] = 1; raw.loc[bx, "qty_issued"] = 1  # 상자(10EA) 단위
    raw.loc[idx(0.01), "qty_req"] = np.nan                                              # 수량 결측
    raw.loc[idx(0.005), "date"] = "2027-01-15"                                          # 미래 일자
    raw.loc[idx(0.005), "nsn"] = "9999-00-000-0000"                                     # 미등록 NSN
    typo = [i for i in idx(0.01) if not raw.at[i, "emergency"]]
    raw.loc[typo, "qty_req"] = raw.loc[typo, "qty_req"] * 10                            # 0 하나 더 입력한 오타
    raw = pd.concat([raw, raw.sample(60, random_state=1)], ignore_index=True)          # 중복 전송
    print(f"원시 청구 기록 {len(raw)}건 (가상), 전산 장애 월 {sorted(OUTAGE)}")


    # 2) 정제 규칙 10종
    def fmt_nsn(s):
        # [TODO : 학생 실습 2] 결측치 보정, 계절성 수요 분리 및 수리부속 품목코드(NSN) 정제 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    log = {}
    df = raw.copy()
    before = len(df); df = df.drop_duplicates("req_id"); log["R1 중복 청구번호 제거"] = before - len(df)
    em0 = int(df["emergency"].sum())
    fixed = df["nsn"].map(fmt_nsn); log["R2 NSN 형식 정규화(공백·하이픈)"] = int((fixed != df["nsn"]).sum()); df["nsn"] = fixed
    unk = ~df["nsn"].isin(master); log["R3 미등록 NSN 격리"] = int(unk.sum()); em_drop = int(df.loc[unk, "emergency"].sum()); df = df[~unk]
    d2 = df["date"].str.replace(".", "-", regex=False); log["R4 날짜 형식 통일"] = int((d2 != df["date"]).sum()); df["date"] = pd.to_datetime(d2)
    fut = df["date"] > "2025-12-31"; log["R5 미래 일자 제거"] = int(fut.sum()); em_drop += int(df.loc[fut, "emergency"].sum()); df = df[~fut]
    neg = df["qty_req"] < 0; log["R6 음수 수량 부호 보정"] = int(neg.sum()); df.loc[neg, "qty_req"] = -df.loc[neg, "qty_req"]
    na = df["qty_req"].isna(); log["R7 청구 수량 결측 -> 불출 수량으로 보완"] = int(na.sum()); df.loc[na, "qty_req"] = df.loc[na, "qty_issued"]
    bxm = df["uoi"] == "BX"; log["R8 단위 환산(BX=10EA)"] = int(bxm.sum())
    df.loc[bxm, ["qty_req", "qty_issued"]] = df.loc[bxm, ["qty_req", "qty_issued"]] * 10; df.loc[bxm, "uoi"] = "EA"
    over = df["qty_issued"] > df["qty_req"]; log["R9 불출>청구 -> 청구 수량으로 제한"] = int(over.sum())
    df.loc[over, "qty_issued"] = df.loc[over, "qty_req"]
    cap = df.groupby("nsn")["qty_req"].transform(lambda s: s.median() * 5)
    out = (df["qty_req"] > cap) & ~df["emergency"]
    log["R10 오타성 이상치 보정(긴급 불출은 보존)"] = int(out.sum()); df.loc[out, "qty_req"] = (df.loc[out, "qty_req"] / 10).round()
    df["qty_req"] = df["qty_req"].astype(int)

    # 3) 월별 집계 + 결측 월 계절성 보간(R11): 작년 같은 달 x (최근 수준 / 작년 수준)
    df["ym"] = df["date"].dt.strftime("%Y-%m")
    monthly = df.pivot_table(index="ym", columns="nsn", values="qty_req", aggfunc="sum")
    all_m = [f"{y}-{m:02d}" for y in (2024, 2025) for m in range(1, 13)]
    monthly = monthly.reindex(all_m)
    prev = hist[hist["t"].between(12, 23)].assign(nsn=lambda x: x["item_id"].map(nsn_of)).pivot(index="month", columns="nsn", values="demand")
    seasonal_fill = monthly.copy()
    for ym in OUTAGE:
        i = all_m.index(ym)
        ratio = monthly.iloc[max(0, i - 3):i].mean() / prev.loc[[(i - k) % 12 + 1 for k in (1, 2, 3)]].mean().replace(0, np.nan)
        seasonal_fill.loc[ym] = (prev.loc[int(ym[-2:])] * ratio.fillna(1)).round()
    linear_fill = monthly.interpolate()
    monthly = monthly.fillna(0)
    seasonal_fill = seasonal_fill.fillna(0)
    log["R11 결측 월 계절성 보간"] = len(OUTAGE) * monthly.shape[1]
    truth = hist[hist["t"] >= 24].assign(nsn=lambda x: x["item_id"].map(nsn_of)).pivot(index="ym", columns="nsn", values="demand")
    err_s = np.abs(seasonal_fill.loc[sorted(OUTAGE)] - truth.loc[sorted(OUTAGE)]).values.mean()
    err_l = np.abs(linear_fill.loc[sorted(OUTAGE)] - truth.loc[sorted(OUTAGE)]).values.mean()

    print("\n[정제 규칙 적용 결과]")
    for k, v in log.items():
        print(f"  {k:32s} {v:5d}건")
    print(f"결측 월 보간 오차(MAE): 계절성 보간 {err_s:.2f}개 / 단순 선형 보간 {err_l:.2f}개")

    # 4) 정합성 검증
    checks = {"청구번호 유일": df["req_id"].is_unique, "NSN 형식": df["nsn"].str.match(r"^\d{4}-\d{2}-\d{3}-\d{4}$").all(),
              "마스터 등록 NSN": df["nsn"].isin(master).all(), "수량 양수": (df["qty_req"] > 0).all(),
              "불출<=청구": (df["qty_issued"] <= df["qty_req"]).all(), "단위 EA 통일": (df["uoi"] == "EA").all(),
              "미래 일자 없음": (df["date"] <= "2025-12-31").all(), "결측 월 없음": not seasonal_fill.isna().any().any(),
              "긴급 불출 보존": int(df["emergency"].sum()) == em0 - em_drop,
              "월 합계 = 기록 합계": int(monthly.values.sum()) == int(df["qty_req"].sum())}
    rate = sum(checks.values()) / len(checks)
    print(f"\n정합성 검증 {sum(checks.values())}/{len(checks)} 통과({rate:.0%}) - 목표 100%: {'달성' if rate == 1 else '미달'}, 적용 규칙 {len(log)}종(목표 10종 이상)")

    # 5) 전후 기술통계 비교
    stat = pd.DataFrame({"정제 전": pd.to_numeric(raw["qty_req"], errors="coerce").describe(), "정제 후": df["qty_req"].describe()}).round(2)
    print("\n[청구 수량 기술통계 전후 비교]")
    print(stat.T.to_string())

    # 6) 계절성 분리(고전적 분해): 추세 = 12개월 중심 이동평균, 계절지수 = 월별 (실적/추세) 평균
    bat = items[items["fsc"] == "6140"].sort_values("base")["nsn"].iloc[-1]      # 수요가 가장 많은 축전지 품목
    series = pd.concat([truth.iloc[:0], seasonal_fill])[bat].astype(float)
    full = hist[hist["item_id"] == items.set_index("nsn").loc[bat, "item_id"]].set_index("ym")["demand"].astype(float)
    full.loc[series.index] = series
    trend = full.rolling(12, center=True).mean()
    sidx = (full / trend).groupby([int(i[-2:]) for i in full.index]).mean()
    sidx = sidx / sidx.mean()
    print(f"\n축전지 품목 {bat} 계절지수(1=평균):", " ".join(f"{m}월 {v:.2f}" for m, v in sidx.items()))
    print("해석: 겨울(12~2월) 축전지 수요 증가와 훈련 월(3·8·10월) 증가가 함께 나타난다. 훈련 효과는 미션 4에서 별도 피처로 분리한다.")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(full.values, label="demand"); ax[0].plot(trend.values, label="trend (12M centered MA)")
    ax[0].set_title(f"Battery item {bat}: demand vs trend"); ax[0].set_xlabel("month index"); ax[0].legend()
    ax[1].bar(sidx.index, sidx.values); ax[1].axhline(1, c="k", ls="--"); ax[1].set_title("Seasonal index by month"); ax[1].set_xlabel("month")
    plt.tight_layout(); plt.show()


## 실습 기록과 제출

- 마지막 **통합 산출물·학습기록 입력 화면**을 실행합니다.
- 각 산출물 탭에서 직접 내용을 작성하거나 기존 파일을 불러와 수정하고 개별 저장하세요.
- 학습기록 탭에서 실제 AI 질문·답변, 출력·자료 대조 근거, 검토 3건 이상 및 수정 내용, 최종 판단과 미달·미측정·미실시 상태를 기록합니다.
- 저장 후 ZIP 생성 및 다운로드 버튼을 순서대로 사용하세요.

**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다.

저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 2
        DELIVERABLE_PATHS = ['forms/MNT02_M02_forms.md', 'reports/mnt02_cleansing_quality.md', 'src/etl/mnt02_data_cleansing.py']
        PROJECT_NAME = 'MNT02'
        NOTEBOOK_ROLE = '학생용 Student'
        WORK_ROOT = Path(MNT02_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
                    print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
